# Quantum State Certification — Data for Figure 2
This script finds the optimal layout on IBM's quantum computer to prepare the best possible W state and logs the data. On the paper, it was run on IBM Quebec

In [7]:
import numpy as np
from collections import defaultdict
from qiskit import QuantumCircuit, transpile
from qiskit.circuit import Parameter
from qiskit.quantum_info import SparsePauliOp
from qiskit_ibm_runtime import QiskitRuntimeService, EstimatorV2 as Estimator
import json
import time

In [ ]:
service = QiskitRuntimeService(channel='ibm_quantum_platform', token='YOUR TOKEN', instance="YOUR INSTANCE") 
backend = service.backend(..., use_fractional_gates=True) # fill in parameters

In [9]:
TOP_K = 10          # Number of top chains to show
CHAIN_LENGTH = 7   # Length of the qubit chain


# ==============================================================
# 1. Extract RX, RZZ, and readout errors
# ==============================================================

def collect_rx_errors_all(backend):
    props = backend.properties()
    rx_errors = {}

    for gate in props.gates:
        if gate.gate.lower() != "rx":
            continue
        if len(gate.qubits) != 1:
            continue

        q = gate.qubits[0]
        for p in gate.parameters:
            if p.name == "gate_error":
                rx_errors[q] = p.value
                break

    return rx_errors


def collect_rzz_errors_all(backend):
    props = backend.properties()
    rzz_errors = {}

    for gate in props.gates:
        if gate.gate.lower() != "rzz":
            continue
        if len(gate.qubits) != 2:
            continue

        q0, q1 = sorted(gate.qubits)
        for p in gate.parameters:
            if p.name == "gate_error":
                rzz_errors[(q0, q1)] = p.value
                break

    return rzz_errors


def collect_readout_errors_all(backend):
    """Return dict {qubit: readout_error}"""
    props = backend.properties()
    readout_errors = {}

    for q, qubit_info in enumerate(props.qubits):
        # qubit_info is a list of NamedTuples, find 'readout_error'
        for param in qubit_info:
            if param.name == "readout_error":
                readout_errors[q] = param.value
                break

    return readout_errors



# ==============================================================
# 2. Build graph from coupling map
# ==============================================================

def build_graph(backend):
    graph = defaultdict(list)
    for q0, q1 in backend.configuration().coupling_map:
        graph[q0].append(q1)
        graph[q1].append(q0)
    return graph


# ==============================================================
# 3. DFS to find all unique paths of length k
# ==============================================================

def find_all_unique_paths_length_k(graph, k):
    """Return list of all unique simple paths of length k."""
    unique_paths = set()

    def dfs(path):
        if len(path) == k:
            # store in canonical order to avoid reversed duplicates
            canonical = tuple(path) if path[0] < path[-1] else tuple(reversed(path))
            unique_paths.add(canonical)
            return

        last = path[-1]
        for nxt in graph[last]:
            if nxt in path:
                continue
            path.append(nxt)
            dfs(path)
            path.pop()

    for node in graph:
        dfs([node])

    return [list(p) for p in unique_paths]


# ==============================================================
# 4. Score path (mean RX + mean RZZ + mean readout)
# ==============================================================

def score_path(path, rx_errors, rzz_errors, readout_errors):
    try:
        rx_vals = [rx_errors[q] for q in path]
        read_vals = [readout_errors[q] for q in path]
    except KeyError:
        return np.inf

    rzz_vals = []
    for i in range(len(path) - 1):
        q0, q1 = sorted((path[i], path[i+1]))
        if (q0, q1) not in rzz_errors:
            return np.inf
        rzz_vals.append(rzz_errors[(q0, q1)])

    return np.mean(rx_vals) + np.mean(rzz_vals) + np.mean(read_vals)


# ==============================================================
# 5. Main: Find top-K chains
# ==============================================================

def find_top_k_chains(backend, k=TOP_K, chain_length=CHAIN_LENGTH):
    rx_errors = collect_rx_errors_all(backend)
    rzz_errors = collect_rzz_errors_all(backend)
    readout_errors = collect_readout_errors_all(backend)
    graph = build_graph(backend)

    print(f"RX entries: {len(rx_errors)}  |  RZZ entries: {len(rzz_errors)}  |  Readout entries: {len(readout_errors)}")

    paths = find_all_unique_paths_length_k(graph, chain_length)
    print(f"Found {len(paths)} unique {chain_length}-qubit chains\n")

    scored = []
    for path in paths:
        score = score_path(path, rx_errors, rzz_errors, readout_errors)
        if np.isfinite(score):
            scored.append((score, path))

    scored.sort(key=lambda x: x[0])
    return scored[:k]


# ==============================================================
# 6. Run and display
# ==============================================================

best_list = find_top_k_chains(backend, k=TOP_K, chain_length=CHAIN_LENGTH)

print("\n================= TOP CHAINS =================")
for rank, (score, path) in enumerate(best_list, start=1):
    print(f"#{rank:02d}: Score = {score:.3e}   Path = {path}")
print("==============================================\n")


RX entries: 156  |  RZZ entries: 176  |  Readout entries: 156
Found 753 unique 7-qubit chains


================= TOP CHAINS =================
#01: Score = 1.051e-02   Path = [29, 38, 49, 48, 47, 57, 67]
#02: Score = 1.053e-02   Path = [38, 49, 48, 47, 57, 67, 68]
#03: Score = 1.200e-02   Path = [30, 29, 38, 49, 48, 47, 57]
#04: Score = 1.210e-02   Path = [28, 29, 38, 49, 48, 47, 57]
#05: Score = 1.218e-02   Path = [52, 51, 58, 71, 72, 73, 74]
#06: Score = 1.225e-02   Path = [51, 58, 71, 72, 73, 74, 75]
#07: Score = 1.227e-02   Path = [16, 23, 22, 21, 36, 41, 40]
#08: Score = 1.237e-02   Path = [49, 48, 47, 57, 67, 68, 69]
#09: Score = 1.242e-02   Path = [112, 113, 119, 133, 132, 131, 138]
#10: Score = 1.244e-02   Path = [53, 52, 51, 58, 71, 72, 73]



## This section transpiles the circuit into native fractional gates on IBM's system  

In [10]:
def p_cnot(qc, q1, q2):
    qc.rx(np.pi/2,q2)
    qc.rzz(np.pi/2, q1, q2)
    p_h(qc, q2)

def p_h(qc, q):
    qc.rz(np.pi/2,q)
    qc.sx(q)
    qc.rz(np.pi/2,q)

def p_cry(qc, q1, q2, theta):
    qc.rx(np.pi/2,q2)
    qc.rzz(theta/2,q1,q2)
    qc.rz(np.pi-(theta/2),q2)
    qc.rx(np.pi/2,q2)


def W_n_transpiled_IBM(n):
    qc=QuantumCircuit(n,n)
    for i in range(0,n-1):
        if i==0 :
            qc.rx(2*np.arccos(1/(np.sqrt(n-i))),0)
        else:
            p_cry(qc,i-1,i, 2*np.arccos(1/(np.sqrt(n-i))))
    for j in range(1,n):
        p_cnot(qc,n-j-1,n-j)
    qc.x(0)
    #U=Operator(qc).data
    # qc.measure_all()
    return qc

In [11]:
def w_generator_hamiltonian_H1_over_n(n: int, delta=1.0) -> SparsePauliOp:
    """
    Constructs H_W' = (1/n) * H1 + H2, with no delta parameter.

    Using:
        H1 = -sum_{j<k} SWAP_{jk} + C * I,     C = binom(n,2)
        H2 = (P - I)^2, with P = sum_j |1><1|_j = sum_j (I - Z_j)/2  (number operator)

    Expanding gives Pauli form:
        H_W' =
          -(1/(2n)) * sum_{j<k} (X_j X_k + Y_j Y_k)
          + (n-1)/(2n) * sum_{j<k} (Z_j Z_k)
          + (1 - n/2) * sum_j Z_j
          + I * (n^2 - 2n + 3)/4
    """
    if n < 2:
        raise ValueError("Need at least two qubits.")

    paulis, coeffs = [], []

    # Pair terms over j<k
    xx_yy_coeff = -(1.0 / (2.0 * n))
    zz_coeff = (n - 1.0) / (2.0 * n)

    for j in range(n):
        for k in range(j + 1, n):
            # XX
            term = ["I"] * n
            term[j] = term[k] = "X"
            paulis.append("".join(term))
            coeffs.append(xx_yy_coeff)

            # YY
            term = ["I"] * n
            term[j] = term[k] = "Y"
            paulis.append("".join(term))
            coeffs.append(xx_yy_coeff)

            # ZZ
            term = ["I"] * n
            term[j] = term[k] = "Z"
            paulis.append("".join(term))
            coeffs.append(zz_coeff)

    # Single-Z terms: (1 - n/2) * sum_j Z_j
    z_coeff = 1.0 - 0.5 * n
    for j in range(n):
        term = ["I"] * n
        term[j] = "Z"
        paulis.append("".join(term))
        coeffs.append(z_coeff)

    id_coeff = (n * n - 2.0 * n + 3.0) / 4.0
    paulis.append("I" * n)
    coeffs.append(id_coeff)

    return SparsePauliOp.from_list(list(zip(paulis, coeffs)))

In [ ]:
number_of_shots = 16 * 1024

for i in range(2,21):
    for k in best_list: 
        nb_qubit = i
        H = w_generator_hamiltonian_H1_over_n(n=nb_qubit)
        W_state2 = W_n_transpiled_IBM(nb_qubit)
        qreg = W_state2.qregs[0]
        layout = {}
        for j,qregs in enumerate(qreg):
            layout[qregs]=k[1][j]
            
        qc_transpiled = transpile(
            W_state2,
            backend=backend,
            initial_layout=layout,
            layout_method='trivial', 
            routing_method='none',    
            optimization_level=3
        )

        estimator = Estimator(mode=backend, options={"default_shots": number_of_shots})
        job = estimator.run([(qc_transpiled, H.apply_layout(qc_transpiled.layout))])
        expectation_value_found = job.result()[0].data.evs

        result = {
            "nb_qubit": nb_qubit,
            "prev_energy": float(expectation_value_found),
            "shots": number_of_shots,
        }


        with open(f"/w_state_{time.time()}_{nb_qubit}.json", "w") as f:
            json.dump(result, f, indent=4)

In [13]:
result

{'nb_qubit': 3, 'prev_energy': 0.030136949665239887, 'shots': 4096}